# 👗 Week 2: Data Acquisition & Preprocessing
## Fashion & Apparel Industry

> **Goal:** Demonstrate a complete end-to-end preprocessing pipeline on a synthetic fashion dataset.

---

## 0. Setup & Imports

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.preprocessing import (
    validate_schema, handle_missing_values, remove_duplicates,
    handle_outliers, normalize_features, encode_categoricals, export_processed
)
from src.feature_engineering import engineer_all_features
from src.utils import summarize_dataframe, generate_missing_value_report

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.set_option('display.max_columns', 20)

print('All imports successful!')

## 1. Generate Synthetic Fashion Dataset

In [ ]:
np.random.seed(42)
N = 1000

df_raw = pd.DataFrame({
    'product_id': range(1, N + 1),
    'product_name': [f'FashionItem_{i:04d}' for i in range(1, N + 1)],
    'category': np.random.choice(['Tops', 'Bottoms', 'Dresses', 'Footwear', 'Accessories', 'Outerwear'], N),
    'gender': np.random.choice(['Male', 'Female', 'Unisex'], N, p=[0.3, 0.5, 0.2]),
    'price': np.random.exponential(scale=60, size=N).clip(5, 500),
    'discount_pct': np.random.uniform(0, 70, N),
    'rating': np.random.normal(3.8, 0.8, N).clip(1, 5),
    'reviews_count': np.random.randint(0, 10000, N),
    'stock': np.random.randint(0, 2000, N),
    'season': np.random.choice(['Spring', 'Summer', 'Autumn', 'Winter'], N),
    'brand': np.random.choice(['BrandA', 'BrandB', 'BrandC', 'BrandD', 'BrandE',
                               'BrandF', 'BrandG', 'BrandH', 'BrandI', 'BrandJ'], N),
    'color': np.random.choice(['Black', 'White', 'Blue', 'Red', 'Green', 'Yellow', 'Pink', 'Grey', 'Brown', 'Orange'], N),
})

# Inject missing values
for col, frac in [('price', 0.08), ('rating', 0.12), ('gender', 0.05), ('color', 0.1), ('discount_pct', 0.04)]:
    idx = np.random.choice(N, size=int(N * frac), replace=False)
    df_raw.loc[idx, col] = np.nan

# Inject duplicates
df_raw = pd.concat([df_raw, df_raw.iloc[:50]], ignore_index=True)

# Inject outliers
df_raw.loc[5, 'price'] = 9999
df_raw.loc[10, 'price'] = -100
df_raw.loc[20, 'rating'] = 100

print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head()

## 2. Exploratory Overview

In [ ]:
summarize_dataframe(df_raw)

In [ ]:
# Missing value report
missing_report = generate_missing_value_report(df_raw)
print('Missing Value Report:')
print(missing_report)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Price distribution (raw — note outliers)
df_raw['price'].dropna().hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Raw Price Distribution', fontsize=12)
axes[0].set_xlabel('Price ($)')

# Category distribution
df_raw['category'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Product Categories', fontsize=12)
axes[1].tick_params(axis='x', rotation=30)

# Rating distribution
df_raw['rating'].dropna().hist(bins=30, ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Rating Distribution', fontsize=12)

plt.suptitle('Raw Data — Exploratory Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_raw.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Step-by-Step Preprocessing

### Step 1: Schema Validation

In [ ]:
REQUIRED_COLS = ['product_id', 'category', 'price', 'rating']
df = validate_schema(df_raw.copy(), required_columns=REQUIRED_COLS)
print('Schema validation passed!')

### Step 2: Handle Missing Values

In [ ]:
df = handle_missing_values(df, numeric_strategy='median', categorical_strategy='mode')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

### Step 3: Remove Duplicates

In [ ]:
df = remove_duplicates(df, subset=['product_id'])
print(f'Shape after deduplication: {df.shape}')

### Step 4: Outlier Treatment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['price'].plot(kind='box', ax=axes[0], title='Before Outlier Treatment — Price')

df = handle_outliers(df, columns=['price', 'rating', 'reviews_count'], method='iqr', action='clip')

df['price'].plot(kind='box', ax=axes[1], title='After Outlier Treatment — Price')
plt.tight_layout()
plt.show()

### Step 5: Categorical Encoding

In [ ]:
# Encode low-cardinality columns with One-Hot, high-cardinality with Label
df = encode_categoricals(
    df,
    onehot_cols=['gender', 'season', 'category'],
    label_cols=['brand', 'color', 'product_name']
)
print(f'Shape after encoding: {df.shape}')
print(f'New one-hot columns: {[c for c in df.columns if "_" in c][:10]}')

### Step 6: Normalization

In [ ]:
# Drop non-numeric columns before scaling
num_to_scale = ['price', 'discount_pct', 'rating', 'reviews_count', 'stock']
num_to_scale = [c for c in num_to_scale if c in df.columns]

df = normalize_features(df, columns=num_to_scale, method='minmax')
print('Normalized column ranges:')
print(df[num_to_scale].describe().loc[['min', 'max']].round(4))

## 4. Feature Engineering

In [ ]:
# Reload pre-encoded df for feature engineering (to work on unscaled values)
# Feature engineering best done before scaling
df_fe = df_raw.copy()
df_fe = handle_missing_values(df_fe)
df_fe = remove_duplicates(df_fe, subset=['product_id'])
df_fe = handle_outliers(df_fe, columns=['price', 'rating'], method='iqr', action='clip')
df_fe = engineer_all_features(df_fe)

print('Engineered features added:')
new_cols = ['price_tier', 'discount_category', 'season_sin', 'season_cos',
            'popularity_score', 'effective_price', 'revenue_estimate']
print(df_fe[[c for c in new_cols if c in df_fe.columns]].head(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price tier distribution
df_fe['price_tier'].value_counts().plot(kind='bar', ax=axes[0,0], color='steelblue', edgecolor='white')
axes[0,0].set_title('Products by Price Tier', fontsize=12)
axes[0,0].tick_params(axis='x', rotation=0)

# Discount category
df_fe['discount_category'].value_counts().plot(kind='bar', ax=axes[0,1], color='coral', edgecolor='white')
axes[0,1].set_title('Discount Categories', fontsize=12)
axes[0,1].tick_params(axis='x', rotation=30)

# Popularity score
df_fe['popularity_score'].hist(bins=30, ax=axes[1,0], color='mediumseagreen', edgecolor='white')
axes[1,0].set_title('Popularity Score Distribution', fontsize=12)

# Revenue estimate by category
df_fe.groupby('category')['revenue_estimate'].sum().sort_values(ascending=True).plot(
    kind='barh', ax=axes[1,1], color='mediumpurple', edgecolor='white')
axes[1,1].set_title('Revenue Estimate by Category', fontsize=12)

plt.suptitle('Engineered Features Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Export Processed Data

In [ ]:
paths = export_processed(df_fe, filename='fashion_processed_week2', formats=['csv'])
print('Export complete!')
for fmt, path in paths.items():
    print(f'  {fmt.upper()}: {path}')

print(f'\nFinal dataset shape: {df_fe.shape}')
print(f'Columns: {list(df_fe.columns)}')

## 6. Pipeline Summary

In [ ]:
summary = pd.DataFrame([
    ['Raw Dataset', df_raw.shape[0], df_raw.shape[1], df_raw.isnull().sum().sum(), df_raw.duplicated().sum()],
    ['After Missing Value Handling', '-', '-', 0, df_raw.duplicated().sum()],
    ['After Deduplication', df_fe.shape[0], '-', 0, 0],
    ['After Outlier Treatment', df_fe.shape[0], df_fe.shape[1], 0, 0],
    ['After Feature Engineering', df_fe.shape[0], df_fe.shape[1], 0, 0],
], columns=['Stage', 'Rows', 'Columns', 'Missing Values', 'Duplicates'])

print('\n=== PREPROCESSING PIPELINE SUMMARY ===')
print(summary.to_string(index=False))
print('=======================================')